# 딥러닝 기반 단어중의성 해소

1. WordNet에서 의미 후보 수집

2. 문맥(입력 문장)과 의미 설명(gloss)을 임베딩

3. 코사인 유사도 계산

4. 가장 가까운 의미 선택

- transformers + torch → 딥러닝 모델 로딩/추론

- nltk.corpus.wordnet → 단어 의미 후보(gloss) 가져오기

- nltk.download → 필수 리소스 다운로드 (처음 실행 시만)

In [2]:
# ============================================
# Neural WSD Demo (BERT/MiniLM 기반 코사인 유사도)
# ============================================

# (필요시) 라이브러리 설치
# Colab이나 새 환경에서 실행 시 아래 주석을 풀고 설치해야 함
# !pip install -q transformers torch nltk

# 수학 계산을 위한 라이브러리
import math
import numpy as np

# 파이토치(PyTorch): 딥러닝 모델을 다루기 위한 라이브러리
import torch

# HuggingFace Transformers에서 토크나이저와 사전학습된 모델 불러오기
from transformers import AutoTokenizer, AutoModel

# NLTK: 자연어처리용 파이썬 라이브러리
import nltk
# WordNet: 영어 단어 사전(동의어, 반의어, 정의 등 포함)
from nltk.corpus import wordnet as wn

# NLTK 리소스 다운로드 (처음 한 번만 실행하면 됨)
nltk.download('wordnet')   # WordNet 사전
nltk.download('omw-1.4')   # WordNet 다국어 번역 지원 데이터

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

토크나이저: 문장을 모델이 처리 가능한 숫자(토큰 id)로 변환

모델: 토큰을 받아서 의미를 담은 벡터(embedding)를 만들어줌

eval(): 학습이 아닌 추론 모드로 실행

In [3]:
# 1) 모델 로딩

# 사용할 사전학습 모델 이름 지정
# 'sentence-transformers/all-MiniLM-L6-v2'는 가볍고 빠르면서도
# 문장을 벡터로 변환(임베딩)할 때 널리 쓰이는 모델
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# 토크나이저 불러오기
# 입력 문장을 모델이 이해할 수 있는 토큰(id) 시퀀스로 변환
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 사전학습된 모델 불러오기
# 입력된 토큰을 받아 임베딩(벡터 표현)을 생성
model = AutoModel.from_pretrained(MODEL_NAME)

# 모델을 '학습 모드'가 아닌 '추론 모드'로 전환
# Dropout 같은 학습용 기능을 끄고, 예측만 수행
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


mean pooling: 모든 단어 벡터를 평균 내어 문장의 의미를 대표하는 벡터로 만듦

encode: 문장을 BERT 계열 모델에 넣어 정규화된 벡터로 변환

cosine: 두 벡터가 얼마나 비슷한지(각도가 얼마나 가까운지)를 계산

In [4]:
# 2) 유틸 함수: 문장을 임베딩(벡터)으로 변환하는 도구들

# 평균 풀링(mean pooling)으로 문장 임베딩 만들기
# (BERT의 [CLS] 토큰 대신, 모든 토큰 벡터의 평균을 사용)
def mean_pooling(model_output, attention_mask):
    token_embeds = model_output.last_hidden_state          # (B, T, H): 배치×토큰×히든차원
    # attention_mask: 실제 단어 위치=1, 패딩 위치=0
    # 크기를 토큰 임베딩과 맞춰서 확장
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    # 마스크된 토큰만 더한 값
    sum_embeds = (token_embeds * input_mask_expanded).sum(dim=1)
    # 실제 단어 개수 (0으로 나누지 않도록 최소값 보정)
    sum_mask = input_mask_expanded.sum(dim=1).clamp(min=1e-9)
    # 평균 내어 문장 벡터 생성
    return sum_embeds / sum_mask                           # (B, H): 배치×히든차원

# 텍스트 리스트를 받아서 문장 임베딩 벡터(np.array)로 변환
def encode(texts):
    """텍스트 리스트 → 문장 임베딩 (np.array)"""
    # 토큰화 + 패딩 + 텐서 변환
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():  # 학습 모드가 아니라 추론 모드로 실행
        out = model(**enc)  # 모델에 입력 전달 → 토큰별 임베딩 얻음
        sent_emb = mean_pooling(out, enc["attention_mask"])  # 평균 풀링으로 문장 임베딩 생성
        # 임베딩을 정규화(normalize) → 코사인 유사도 계산이 쉬워짐
        sent_emb = torch.nn.functional.normalize(sent_emb, p=2, dim=1)
    return sent_emb.cpu().numpy()  # 넘파이 배열로 반환

# 두 벡터 간 코사인 유사도 계산 함수
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

lemmas: 해당 의미(sense)와 연결된 동의어 단어들

gloss: WordNet이 제공하는 정의(사전적 설명)

examples(): 해당 의미가 실제 문장에서 쓰인 예시

이 세 가지를 합쳐 “의미 설명 문장”을 만들면, BERT 같은 모델에 넣어 문맥과 비교할 수 있음

In [5]:
# 3) WordNet 의미 후보(synset)를 사람이 읽을 수 있는 문장으로 변환
# - 정의(gloss), 예문(example), 동의어(lemma)들을 합쳐 하나의 텍스트로 만듦
def sense_to_text(syn):
    # 해당 synset의 동의어(lemma) 최대 4개까지 추출, 밑줄(_)은 공백으로 바꿈
    lemmas = ", ".join(l.name().replace("_", " ") for l in syn.lemmas()[:4])

    # synset의 정의(gloss)
    gloss = syn.definition()

    # 예문(example) 중 첫 번째만 사용, 있으면 앞에 "ex:" 붙임
    ex = (" | ex: " + syn.examples()[0]) if syn.examples() else ""

    # 최종 문자열 조합
    # 예: "bank.n.09 :: (bank, depository financial institution) — a financial institution ... | ex: she cashed a check at the bank"
    return f"{syn.name()} :: ({lemmas}) — {gloss}{ex}"

문맥 문장과 각 의미의 설명 텍스트를 임베딩으로 변환

코사인 유사도를 계산해 가장 가까운 의미를 선택

결과로 최적 의미(1개)와 상위 후보(topk)를 함께 반환

In [6]:
# 4) Neural WSD: 문맥 문장 vs 각 의미(gloss 문장)의 임베딩 유사도 비교
def neural_wsd(target_word: str, sentence: str, topk=5):
    # WordNet에서 대상 단어(target_word)의 모든 synset(후보 의미) 가져오기
    synsets = wn.synsets(target_word)
    if not synsets:                 # 후보 의미가 없으면 None 반환
        return None, []

    # 1) 입력 문장의 임베딩 구하기
    # encode 함수로 문장을 벡터화 (BERT 기반 문맥 임베딩)
    ctx_vec = encode([sentence])[0]

    # 2) 각 synset을 설명하는 텍스트(gloss+동의어+예문) 만들기
    sense_texts = [sense_to_text(s) for s in synsets]

    # 3) 각 의미 설명 텍스트를 임베딩 벡터로 변환
    sense_vecs = encode(sense_texts)

    # 4) 문맥 임베딩(ctx_vec)과 의미 임베딩(sense_vecs)의 코사인 유사도 계산
    scores = []
    for s, st, v in zip(synsets, sense_texts, sense_vecs):
        # (synset 객체, 텍스트 설명, 유사도 점수) 형태로 저장
        scores.append((s, st, cosine(ctx_vec, v)))

    # 5) 유사도 점수 기준으로 내림차순 정렬 (가장 비슷한 의미가 맨 앞)
    scores.sort(key=lambda x: x[2], reverse=True)

    # 가장 높은 점수를 받은 의미(synset)와, 상위 topk개 후보 목록 반환
    return scores[0][0], scores[:topk]

Top candidates: 문맥과 가장 비슷한 후보 의미들 (점수와 함께 순위 출력)

Predicted sense: 그중 1위 → 최종 선택된 의미

출력 형식을 보기 좋게 만들어 디버깅 + 설명용으로 활용 가능

In [7]:
# 5) 보기 좋은 출력 함수
def pretty_print(sentence, target, top_scores):
    # 입력 문장과 대상 단어 출력
    print("📌 Sentence:", sentence)
    print("🔎 Target  :", target)
    print("-" * 80)

    # 상위 후보 의미(top_scores) 출력
    print("Top candidates (by cosine similarity):")
    for rank, (syn, text, sc) in enumerate(top_scores, 1):
        # 순위, synset 이름, 품사, 점수 출력
        print(f"{rank:>2}. {syn.name():<20} | POS={syn.pos()} | score={sc:0.4f}")
        # synset 설명(동의어, 정의, 예문)도 함께 보여줌
        print(f"    ↳ {text}")

    print("-" * 80)

    # 최종적으로 선택된 의미(가장 점수 높은 것) 출력
    print(f"✅ Predicted sense: {top_scores[0][0].name()}  —  {top_scores[0][0].definition()}")
    print()  # 줄바꿈

동일한 단어라도 문맥에 따라 다른 의미로 분류되는 과정을 직접 확인할 수 있음.

bank → 은행(금융기관) / 강둑, bat → 박쥐 / 방망이 처럼 WSD의 필요성을 직관적으로 보여주는 데모.

In [8]:
# 6) 데모: 같은 단어가 문맥에 따라 어떻게 다른 의미로 해석되는지 확인

# 예시 문장 집합
examples = [
    ("bank", "I deposited cash at the bank near my office."),   # bank → 금융기관(은행)
    ("bank", "We had a picnic on the bank of the river."),      # bank → 강둑
    ("bat",  "The bat flew out of the cave at dusk."),          # bat → 박쥐(동물)
    ("bat",  "He swung the bat and hit a home run."),           # bat → 방망이(야구 배트)
]

# 각 예제 문장에 대해 WSD 실행
for word, sent in examples:
    # neural_wsd 함수 실행 → (최적 의미, 상위 후보 리스트) 반환
    best, top = neural_wsd(word, sent, topk=5)

    # 보기 좋은 출력 함수 호출 → 후보 의미와 최종 선택 결과 출력
    pretty_print(sent, word, top)

📌 Sentence: I deposited cash at the bank near my office.
🔎 Target  : bank
--------------------------------------------------------------------------------
Top candidates (by cosine similarity):
 1. deposit.v.02         | POS=v | score=0.4796
    ↳ deposit.v.02 :: (deposit, bank) — put into a bank account | ex: She deposits her paycheck every month
 2. bank.v.01            | POS=v | score=0.4281
    ↳ bank.v.01 :: (bank) — tip laterally | ex: the pilot had to bank the aircraft
 3. bank.v.03            | POS=v | score=0.4090
    ↳ bank.v.03 :: (bank) — do business with a bank or keep an account at a bank | ex: Where do you bank in this town?
 4. savings_bank.n.02    | POS=n | score=0.3511
    ↳ savings_bank.n.02 :: (savings bank, coin bank, money box, bank) — a container (usually with a slot in the top) for keeping money at home | ex: the coin bank was empty
 5. bank.v.05            | POS=v | score=0.3302
    ↳ bank.v.05 :: (bank) — be in the banking business
----------------------------